# ForkWise — Part 1: Provision Infrastructure

**What this does:** Creates 3 VMs on Chameleon KVM@TACC using Terraform.

**What you need before running:**
1. Chameleon account with access to KVM@TACC
2. SSH keypair `forkwise-key` already uploaded to Chameleon (Compute → Key Pairs)
3. `clouds.yaml` already in `infra/tf/kvm/` (the repo includes one)

**After this notebook completes**, it will print:
- The floating IP of node1
- Instructions to SSH in and upload your private key
- All Part 2 commands to run on node1

Run cells **top to bottom**. Each cell is re-runnable.

## Step 0: Install Dependencies

In [ ]:
import subprocess, sys, os
from pathlib import Path

LOCAL_BIN = Path.home() / ".local" / "bin"
LOCAL_BIN.mkdir(parents=True, exist_ok=True)

if str(LOCAL_BIN) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{LOCAL_BIN}:{os.environ['PATH']}"


def sh(cmd, check=True):
    print(f">>> {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip()[-800:])
    if r.stderr.strip():
        print(r.stderr.strip()[-400:])
    if check and r.returncode != 0:
        raise RuntimeError(f"Failed: {cmd}")
    return r


def is_installed(name):
    return subprocess.run(
        f"which {name}", shell=True, capture_output=True
    ).returncode == 0


# ── Terraform ──
if not is_installed("terraform"):
    print("Installing Terraform...")
    tf_ver = "1.9.8"
    sh(f"curl -fsSL -o /tmp/tf.zip https://releases.hashicorp.com/terraform/{tf_ver}/terraform_{tf_ver}_linux_amd64.zip")
    sh(f"cd /tmp && unzip -o tf.zip && mv terraform {LOCAL_BIN}/terraform && chmod +x {LOCAL_BIN}/terraform && rm tf.zip")
else:
    print("Terraform already installed")

# ── Verify ──
print()
for tool in ["terraform", "ssh"]:
    found = is_installed(tool)
    status = "OK" if found else "MISSING"
    print(f"  [{status}]  {tool}")

## Step 1: Configure Variables

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd()
TF_DIR = REPO_ROOT / "infra" / "tf" / "kvm"

# ── Change these if needed ──
SUFFIX = "proj01"
KEYPAIR_NAME = "forkwise-key"
IMAGE_NAME = "CC-Ubuntu24.04"
FLAVOR_NAME = "m1.medium"
FLAVOR_ID = ""                    # Set this if using a reservation-backed flavor
SHAREDNET = "sharednet1"
FIP_POOL = "public"
SECURITY_GROUPS = '["default"]'

print(f"Repo root:    {REPO_ROOT}")
print(f"Terraform:    {TF_DIR}")
print(f"Suffix:       {SUFFIX}")
print(f"Keypair:      {KEYPAIR_NAME}")
print(f"Image:        {IMAGE_NAME}")
print(f"Flavor:       {FLAVOR_ID or FLAVOR_NAME}")

## Step 2: Write terraform.tfvars

In [ ]:
tfvars = f'''suffix               = "{SUFFIX}"
key                  = "{KEYPAIR_NAME}"
image_name           = "{IMAGE_NAME}"
flavor_id            = "{FLAVOR_ID}"
flavor_name          = "{FLAVOR_NAME}"
sharednet_name       = "{SHAREDNET}"
floating_ip_pool     = "{FIP_POOL}"
security_group_names = {SECURITY_GROUPS}
'''

(TF_DIR / "terraform.tfvars").write_text(tfvars)
print("Wrote terraform.tfvars:")
print(tfvars)

# Verify clouds.yaml exists
clouds = TF_DIR / "clouds.yaml"
if clouds.exists():
    print("clouds.yaml found")
else:
    print("ERROR: clouds.yaml NOT FOUND - copy it to infra/tf/kvm/ before continuing")

## Step 3: Terraform Init & Plan

In [ ]:
os.chdir(TF_DIR)
os.environ["OS_CLIENT_CONFIG_FILE"] = str(TF_DIR / "clouds.yaml")
os.environ["OS_CLOUD"] = "openstack"

# Clear any stale OS_ vars
for key in list(os.environ):
    if key.startswith("OS_") and key not in ("OS_CLIENT_CONFIG_FILE", "OS_CLOUD"):
        del os.environ[key]

sh("terraform init")
print()
sh("terraform validate")
print()
sh("terraform plan")

## Step 4: Terraform Apply

This creates the 3 VMs and assigns a floating IP to node1. Takes 2-5 minutes.

In [ ]:
r = sh("terraform apply -auto-approve -parallelism=1")

# Extract floating IP from output
ip_result = sh("terraform output -raw floating_ip", check=False)
FLOATING_IP = ip_result.stdout.strip()

if FLOATING_IP:
    print(f"\nFloating IP: {FLOATING_IP}")
else:
    print("\nCould not extract floating IP. Run: terraform output floating_ip")

## Step 5: Next Steps — Part 2 Instructions

In [ ]:
from IPython.display import display, Markdown

assert FLOATING_IP, "FLOATING_IP not set - run Step 4 or set manually: FLOATING_IP = 'x.x.x.x'"

# ══════════════════════════════════════════════════════════
# Fill in your object-store credentials here
# ══════════════════════════════════════════════════════════
OS_ACCESS_KEY = "YOUR_ACCESS_KEY_HERE"
OS_SECRET_KEY = "YOUR_SECRET_KEY_HERE"

instructions = f"""
---

# Part 2: Setup Cluster & Deploy (run on node1)

## A. Upload your SSH key to node1

From your **Windows PowerShell**:

```powershell
scp -i $HOME\.ssh\forkwise_key $HOME\.ssh\forkwise_key cc@{FLOATING_IP}:~/.ssh/id_rsa
```

Then SSH into node1:

```powershell
ssh -i $HOME\.ssh\forkwise_key cc@{FLOATING_IP}
```

Fix key permissions:

```bash
chmod 600 ~/.ssh/id_rsa
```

---

## B. Pre-K8s Setup (run on ALL 3 nodes)

Run these on **node1** first:

```bash
sudo apt-get update && sudo apt-get install -y curl ca-certificates jq python3
sudo systemctl stop ufw 2>/dev/null; sudo systemctl disable ufw 2>/dev/null
echo "br_netfilter" | sudo tee /etc/modules-load.d/k8s.conf
sudo modprobe br_netfilter
cat <<EOF | sudo tee /etc/sysctl.d/99-kubernetes-cri.conf
net.bridge.bridge-nf-call-iptables = 1
net.bridge.bridge-nf-call-ip6tables = 1
net.ipv4.ip_forward = 1
EOF
sudo sysctl --system
```

Then SSH to **node2** and **node3** and run the same commands:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.12
# run the same block above, then exit

ssh -o StrictHostKeyChecking=no cc@192.168.1.13
# run the same block above, then exit
```

---

## C. Install k3s

### C1. Install k3s server on node1:

```bash
curl -sfL https://get.k3s.io | \
  INSTALL_K3S_EXEC="server --write-kubeconfig-mode 644 --node-ip 192.168.1.11 --advertise-address 192.168.1.11 --tls-san 192.168.1.11" \
  sh -
```

### C2. Get the join token:

```bash
K3S_TOKEN=$(sudo cat /var/lib/rancher/k3s/server/node-token)
echo $K3S_TOKEN
```

### C3. Install k3s agent on node2:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.12 \
  "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip 192.168.1.12' sh -"
```

### C4. Install k3s agent on node3:

```bash
ssh -o StrictHostKeyChecking=no cc@192.168.1.13 \
  "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip 192.168.1.13' sh -"
```

### C5. Verify all nodes joined:

```bash
sudo kubectl get nodes
```

Wait until all 3 show `Ready` (1-2 minutes).

---

## D. Post-k3s Setup (on node1)

```bash
mkdir -p ~/.kube
sudo cp /etc/rancher/k3s/k3s.yaml ~/.kube/config
sudo chown $(id -u):$(id -g) ~/.kube/config
sed -i 's|https://127.0.0.1:6443|https://192.168.1.11:6443|' ~/.kube/config

curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
kubectl patch deployment metrics-server -n kube-system --type=json \
  -p='[{{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}}]'

kubectl label node $(hostname) forkwise.io/entrypoint=true --overwrite
kubectl get nodes
```

---

## E. Clone Repo & Copy Manifests (on node1)

```bash
cd ~
git clone https://github.com/HivanshD/ml-sys-ops-project.git
sudo cp -r ~/ml-sys-ops-project/infra/k8s /tmp/serving-k8s
```

---

## F. Build Docker Images (on node1)

### F1. Data images:

```bash
cd ~/ml-sys-ops-project/data
sudo docker build -t forkwise-ingest:local -f Dockerfile.ingest .
sudo docker build -t forkwise-feedback:local -f Dockerfile.feedback .
sudo docker build -t forkwise-batch:local -f Dockerfile.batch .
sudo docker build -t forkwise-generator:local -f Dockerfile.generator .
```

### F2. Serving image:

```bash
cd ~/ml-sys-ops-project/serving
sudo docker build -t subst-serving-onnx:local -f docker/Dockerfile.fastapi_onnx .
```

### F3. Training image:

```bash
cd ~/ml-sys-ops-project
sudo docker build -t forkwise-train:local -f training/docker_nvidia/Dockerfile .
```

### F4. Automation image:

```bash
cd ~/ml-sys-ops-project/infra/automation
sudo docker build -t forkwise-automation:local .
```

### F5. Mealie image:

```bash
cd ~
git clone https://github.com/HivanshD/mealie.git
cd mealie
sudo docker build -t mealie:ml-ui -f docker/Dockerfile .
```

### F6. Import images into k3s containerd:

```bash
for img in forkwise-ingest:local forkwise-feedback:local forkwise-batch:local \
           forkwise-generator:local subst-serving-onnx:local forkwise-train:local \
           forkwise-automation:local mealie:ml-ui; do
  echo "Importing $img..."
  sudo docker save $img | sudo k3s ctr images import -
done
```

---

## G. Update K8s Manifests for Local Images (on node1)

```bash
cd /tmp/serving-k8s

sed -i 's|ghcr.io/itsnotaka/forkwise-ingest:demo|forkwise-ingest:local|g' $(grep -rl 'forkwise-ingest' .)
sed -i 's|ghcr.io/itsnotaka/forkwise-feedback:demo|forkwise-feedback:local|g' $(grep -rl 'forkwise-feedback' .)
sed -i 's|ghcr.io/itsnotaka/forkwise-batch:demo|forkwise-batch:local|g' $(grep -rl 'forkwise-batch' .)
sed -i 's|ghcr.io/itsnotaka/forkwise-generator:demo|forkwise-generator:local|g' $(grep -rl 'forkwise-generator' .)
sed -i 's|ghcr.io/itsnotaka/mealie:ml-ui-amd64|mealie:ml-ui|g' $(grep -rl 'mealie:ml-ui' .)

# Add imagePullPolicy: Never for local images
find . -name '*.yaml' | xargs grep -l ':local' | while read f; do
  sed -i '/:local/a\        imagePullPolicy: Never' "$f"
done
```

---

## H. Deploy Apps (on node1)

### H1. Namespaces and Mealie secret:

```bash
kubectl apply -f /tmp/serving-k8s/apps/mealie/namespace.yaml
kubectl apply -f /tmp/serving-k8s/apps/substitution-serving/namespace.yaml

kubectl create secret generic mealie-credentials \
  --namespace forkwise-app \
  --from-literal=postgres-user=mealie \
  --from-literal=postgres-password=$(openssl rand -base64 20) \
  --from-literal=postgres-db=mealie \
  --from-literal=base-url=http://localhost:9000
```

### H2. Deploy:

```bash
kubectl apply -k /tmp/serving-k8s/apps/substitution-serving
kubectl apply -k /tmp/serving-k8s/apps/mealie

kubectl set image deployment/substitution-serving \
  substitution-serving=subst-serving-onnx:local -n forkwise-serving
kubectl set image cronjob/check-rollback \
  check-rollback=subst-serving-onnx:local -n forkwise-serving

kubectl rollout status deployment/substitution-serving -n forkwise-serving --timeout=180s
kubectl rollout status deployment/mealie-postgres -n forkwise-app --timeout=180s
kubectl rollout status deployment/mealie -n forkwise-app --timeout=240s
```

---

## I. Create Secrets (on node1)

```bash
OS_AK="{OS_ACCESS_KEY}"
OS_SK="{OS_SECRET_KEY}"

for ns in monitoring-proj01 staging-proj01 canary-proj01 production-proj01; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic os-credentials -n $ns \
    --from-literal=OS_ENDPOINT=https://chi.tacc.chameleoncloud.org:7480 \
    --from-literal=OS_ACCESS_KEY=$OS_AK \
    --from-literal=OS_SECRET_KEY=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done

for ns in forkwise-data forkwise-serving; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic s3-credentials -n $ns \
    --from-literal=access-key=$OS_AK \
    --from-literal=secret-key=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done
```

---

## J. Deploy Rollout Stack (on node1)

```bash
kubectl apply -k /tmp/serving-k8s/platform

kubectl set image deployment/automation \
  automation=forkwise-automation:local -n monitoring-proj01

kubectl rollout status deployment/prometheus -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/grafana -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/automation -n monitoring-proj01 --timeout=180s

kubectl exec -n monitoring-proj01 deploy/automation -- \
  curl -fsS -X POST http://127.0.0.1:8080/bootstrap-rollout

for env in staging canary production; do
  kubectl apply -k /tmp/serving-k8s/$env
  kubectl set image deployment/subst-serving \
    subst-serving=subst-serving-onnx:local -n ${{env}}-proj01
  kubectl rollout restart deployment/subst-serving -n ${{env}}-proj01
done
```

---

## K. Deploy Data Workloads (on node1)

```bash
kubectl apply -k /tmp/serving-k8s/apps/forkwise-data

kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml
kubectl logs -n forkwise-data job/forkwise-ingest -f
kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s

kubectl scale deployment/data-generator -n forkwise-data --replicas=1
kubectl patch cronjob batch-pipeline -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
kubectl patch cronjob drift-monitor -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
```

---

## L. Smoke Test (on node1)

```bash
kubectl get nodes
kubectl get pods -A
curl -sf http://127.0.0.1:30080/health && echo "Serving OK" || echo "Serving not ready"
curl -sf http://127.0.0.1:30090 -o /dev/null && echo "Mealie OK" || echo "Mealie not ready"
```

---

## M. Access from Windows (on your local PowerShell)

```powershell
ssh -i $HOME\.ssh\forkwise_key -N -L 9000:127.0.0.1:30090 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 8000:127.0.0.1:30080 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 3000:127.0.0.1:30300 cc@{FLOATING_IP}
ssh -i $HOME\.ssh\forkwise_key -N -L 9090:127.0.0.1:30900 cc@{FLOATING_IP}
```

Then open:
- Mealie: http://localhost:9000
- Serving: http://localhost:8000/health
- Grafana: http://localhost:3000
- Prometheus: http://localhost:9090
"""

display(Markdown(instructions))

## Teardown

Uncomment and run to destroy all VMs. Re-run from Step 3 to recreate.

In [ ]:
# os.chdir(TF_DIR)
# os.environ["OS_CLIENT_CONFIG_FILE"] = str(TF_DIR / "clouds.yaml")
# os.environ["OS_CLOUD"] = "openstack"
# sh("terraform destroy -auto-approve")
# print("All VMs destroyed. Re-run from Step 3 to recreate.")